# OLS y SAR (Tapia et al., 2025): versión para Google Colab

Es la sección de OLS y SAR del código de los autores, con estos cambios para que corra en Colab:

1. **Instalación y *imports*:** se instalan `libpysal` y `spreg` por separado (más liviano que el metapaquete `pysal`) y se importan directamente, sin `from pysal.model import spreg`.
2. **Memoria:** `KNN.from_dataframe(k=700)` guarda 36 millones de vínculos en diccionarios de Python y agota la memoria de Colab. Aquí W se construye directamente como matriz dispersa (mismos 700 vecinos más cercanos por distancia esférica, estandarizada por filas) y se pasa a `spreg` como `WSP`. Se comprobó que W, el I de Moran y el SAR coinciden con los de `libpysal`.
3. **Copias explícitas:** el original hace `drop(..., inplace=True)` sobre filtros del `DataFrame`, lo que da advertencias (y falla con *copy-on-write*). Aquí se usa `.copy()`.
4. **Lista de variables definida una vez** (en el original estaba repetida cuatro veces).
5. **Permutaciones de Moran configurables y I de Moran propio** (fórmula y prueba de permutación de `esda`, sobre la W dispersa): con 999 permutaciones sobre más de 50.000 puntos y 700 vecinos tarda varios minutos.
6. **Datos:** se usan los datos reales de Mendeley (`data_paper_plosone_set.csv`, que trae la columna `set`).

La lógica estadística no cambia: log1p(precio), W de 700 vecinos con distancia esférica estandarizada por filas, OLS con errores de White, SAR por mínimos cuadrados en dos etapas (`GM_Lag`) y ranking por coeficientes estandarizados.

In [ ]:
!pip -q install libpysal spreg
import libpysal, spreg
print('libpysal', libpysal.__version__, '| spreg', spreg.__version__)

## Configuración

In [ ]:
# ---- Elija aquí ----
RUTAS_REAL = ['data_paper_plosone_set.csv', '../../data/real/data_paper_plosone_set.csv']   # trae la columna 'set' (Training / Test)
K_VECINOS = 700
PERMUTACIONES_MORAN = 199   # el original usó 999 (p mínimo 0,001); 199 da p mínimo 0,005
# --------------------


## Preparación de los datos

In [ ]:
import os, time, numpy as np, pandas as pd, scipy.sparse as sp
from scipy import stats
from scipy.spatial import cKDTree
from libpysal import weights

ruta = next((r for r in RUTAS_REAL if os.path.exists(r)), None)
if ruta is None:
    from google.colab import files
    print('No se encontró el archivo. Suba data_paper_plosone_set.csv:')
    subido = files.upload()
    ruta = list(subido.keys())[0]

crudo = pd.read_csv(ruta)

VARIABLES = ['bathrooms', 'bedrooms', 'building_type', 'seller_type', 'total_surface', 'quality_municipality',
             'parking_lots', 'swimpool', 'terrace', 'storage_units', 'new_house', 'dist_subway', 'dist_park',
             'dist_hospital', 'dist_police', 'dist_university', 'dist_penitentiary', 'dist_dump', 'dist_industry',
             'dist_mall', 'photo_bathroom', 'photo_bedroom', 'photo_livingroom', 'photo_diningroom',
             'photo_interior', 'photo_exterior', 'photo_balcony', 'photo_laundryroom', 'photo_floorplans',
             'photo_swimpool', 'simce_4b', 'nearby_schools_3km']   # 32 variables: sin latitud ni longitud
columnas = ['price', 'latitude', 'longitude'] + VARIABLES + ['set']
data = crudo[columnas].copy()

training_data = data.loc[data['set'] == 'Training'].drop(columns='set').copy()
test_data = data.loc[data['set'] == 'Test'].drop(columns='set').copy()
data = data.drop(columns='set')
print(len(data), 'propiedades |', len(training_data), 'entrenamiento |', len(test_data), 'prueba')

## Matriz de pesos W (700 vecinos) y estadístico I de Moran

- `construir_W`: los k vecinos más cercanos por distancia esférica (coordenadas sobre la esfera unitaria, la distancia de cuerda conserva el orden de la distancia de arco), con pesos 1/k por filas.
- `moran_I`: I de Moran con W estandarizada por filas y valor p por permutación, como en `esda.Moran`.

In [ ]:
def construir_W(lat, lon, k):
    la, lo = np.radians(np.asarray(lat)), np.radians(np.asarray(lon))
    xyz = np.column_stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)])
    _, idx = cKDTree(xyz).query(xyz, k=k + 1, workers=-1)      # la primera columna es el propio punto
    idx = idx[:, 1:]
    n = len(la)
    return sp.csr_matrix((np.full(n * k, 1.0 / k), (np.repeat(np.arange(n), k), idx.ravel())), shape=(n, n))

def moran_I(valores, W, permutaciones=199, semilla=0):
    z = np.asarray(valores, dtype=float)
    z = z - z.mean()
    denom = float(z @ z)
    I = float(z @ (W @ z)) / denom           # con W por filas, n / S0 = 1
    rng = np.random.default_rng(semilla)
    sim = np.array([float((zp := rng.permutation(z)) @ (W @ zp)) / denom for _ in range(permutaciones)])
    mayores = int((sim >= I).sum())
    mayores = min(mayores, permutaciones - mayores)       # prueba de una cola, igual que esda
    return I, (mayores + 1.0) / (permutaciones + 1.0)

Primero sobre todo el conjunto de datos (log1p del precio).

In [ ]:
t0 = time.time()
W_todo = construir_W(data['latitude'], data['longitude'], K_VECINOS)
I, p = moran_I(np.log1p(data['price']), W_todo, PERMUTACIONES_MORAN)
print(f'Statistic I: {round(I, 4)}')
print(f'P-value: {p}')
print(f'({time.time() - t0:.0f} s)')
del W_todo

## Modelo OLS
Matriz W con los datos de entrenamiento. Se excluyen latitud y longitud porque se usaron para construir W.

In [ ]:
t0 = time.time()
W_train = construir_W(training_data['latitude'], training_data['longitude'], K_VECINOS)
w = weights.WSP(W_train)        # formato disperso que acepta spreg
print(f'W de entrenamiento: {time.time() - t0:.0f} s')

ols_model = spreg.OLS(np.log1p(training_data['price']).values,
                      training_data[VARIABLES].values,
                      name_y='log_price', name_x=VARIABLES, robust='white')
print(ols_model.summary)

In [ ]:
# I de Moran de los residuos del OLS
I, p = moran_I(np.ravel(ols_model.u), W_train, PERMUTACIONES_MORAN)
print(f'Statistic I: {round(I, 4)}')
print(f'P-value: {p}')

### Modelo SAR (rezago espacial, mínimos cuadrados en dos etapas)

In [ ]:
t0 = time.time()
sar_model = spreg.GM_Lag(np.log1p(training_data['price']).values,
                         training_data[VARIABLES].values,
                         w=w, name_y='log_price', name_x=VARIABLES, robust='white')
print(f'GM_Lag: {time.time() - t0:.0f} s')
print(sar_model.summary)

In [ ]:
# I de Moran de los residuos del SAR
I, p = moran_I(np.ravel(sar_model.u), W_train, PERMUTACIONES_MORAN)
print(f'Statistic I: {round(I, 4)}')
print(f'P-value: {p}')
print('\nrho (W_log_price):', float(sar_model.betas[-1][0]))

## Variables más importantes en OLS y SAR
Se estandarizan las variables y se comparan los coeficientes en valor absoluto.

In [ ]:
training_data['ln_price'] = np.log1p(training_data['price'])
cols_z = [c for c in ['ln_price'] + VARIABLES]
training_data_z = training_data[cols_z].dropna().apply(stats.zscore)

ols_model_z = spreg.OLS(training_data_z['ln_price'].values, training_data_z[VARIABLES].values,
                        name_y='log_price', name_x=VARIABLES, robust='white')

coef_ols = np.abs(np.array(ols_model_z.betas).ravel())
ols_importance = pd.DataFrame({'variable': ols_model_z.name_x, 'standarized_coefficient': coef_ols})
ols_importance = (ols_importance[ols_importance['variable'] != 'CONSTANT']
                  .sort_values('standarized_coefficient', ascending=False).reset_index(drop=True))
ols_importance

In [ ]:
sar_model_z = spreg.GM_Lag(training_data_z['ln_price'].values, training_data_z[VARIABLES].values,
                           w=w, name_y='log_price', name_x=VARIABLES, robust='white')

coef_sar = np.abs(np.array(sar_model_z.betas).ravel())[:-1]      # se descarta rho, que va al final
sar_importance = pd.DataFrame({'variable': sar_model_z.name_x, 'standarized_coefficient': coef_sar})
sar_importance = (sar_importance[sar_importance['variable'] != 'CONSTANT']
                  .sort_values('standarized_coefficient', ascending=False).reset_index(drop=True))
sar_importance

### Ranking

In [ ]:
ranking = pd.DataFrame({'ols': ols_importance['variable'], 'sar': sar_importance['variable']})
ranking

## Guardar resultados

In [ ]:
os.makedirs('resultados', exist_ok=True)
resumen = pd.DataFrame({'modelo': ['OLS', 'SAR'],
                        'R2_en_muestra_log': [float(ols_model.r2), float(sar_model.pr2)],
                        'rho': [np.nan, float(sar_model.betas[-1][0])]})
resumen.to_csv('resultados/ols_sar_resumen.csv', index=False)
ranking.to_csv('resultados/ols_sar_ranking.csv', index=False)
print(resumen)
# from google.colab import files; files.download('resultados/ols_sar_ranking.csv')